In [ ]:
from pathlib import Path
import re
import pandas as pd
from shared_config.sqlserver import PROD_SERVER, PYRO_DATABASE
from shared_utils.sqlserver import get_engine, read_table

# DataFrame Settings
pd.set_option('display.max_columns', None)

project_folder = Path.cwd().parent
output_folder = project_folder / 'output'

In [ ]:
prod_engine = get_engine(PROD_SERVER, PYRO_DATABASE)

requests_df = read_table(prod_engine, 'SELECT * FROM printing.request ORDER BY ID DESC;')

In [ ]:
machines_df = requests_df.copy()

# Ensure Build Time is numeric; non-numeric values become NaN
machines_df['Build Time'] = pd.to_numeric(machines_df['Build Time'], errors='coerce')

# Preserve the original machine string for optional reference
machines_df['Original Machine'] = machines_df['Machine']

# Treat blank or null Machine values as an empty string, then split on semicolon
machines_df['Machine'] = (
    machines_df['Machine']
    .fillna('')
    .astype(str)
    .str.split(r'[;,]')
)

# Count valid machines on each original row
machines_df['Machine Count'] = machines_df['Machine'].apply(
    lambda machines: sum(machine.strip() != '' for machine in machines)
)

# Create one row per machine
machines_output = machines_df.explode('Machine')

# Remove extra spaces around machine names
machines_output['Machine'] = machines_output['Machine'].str.strip()

# Keep rows with no machine listed, such as Pending records
# Their Machine value becomes blank
machines_output['Machine Build Time'] = (
    machines_output['Build Time']
    .div(machines_output['Machine Count'].replace(0, pd.NA))
)

# Remove helper column
machines_output = machines_output.drop(columns=['Machine Count'])

# Restore the requested column order
machines_output = machines_output[
    [
        'ID',
        'Machine',
        'Status',
        'Build Time',
        'Machine Build Time',
        'Original Machine',
        'Printing D&T',
        'Print Failure Qty',
        'Legacy Created time',
        'Created',
        'Created_Corrected',
        'Created_Corrected - Year'
    ]
]

# Sort by ID descending
machines_output = machines_output.sort_values('ID', ascending=False)

In [ ]:
cols: list[str] =[
    'ID', 'Print Location', 'Status',
    'Created', 'Created_Corrected', 'Created_Corrected - Year',
    'Quote Accepted D&T', 'Quote Sent D&T',
    'Approved/In Queue D&T',
    'Printing D&T', 'Printing Off Site D&T',
    'Post Processing D&T',
    'On Hold D&T',
    'Complete D&T'
]

stages: list[str] = [col for col in cols if col.endswith('D&T')]
check: list[str] = [col for col in stages if col != 'Complete D&T']
stage_names: list[str] = [stage.removesuffix(' D&T') for stage in check]
order: dict[str, int] = {stage.removesuffix(' D&T'): idx for idx, stage in enumerate(stages)}

status_df: pd.DataFrame = requests_df.copy()
status_df = status_df.loc[status_df['Status'] == 'Complete']
status_df = status_df[cols]
status_df = status_df.dropna(subset=check, how='all')

durations = (
    status_df.melt(id_vars=['ID', 'Print Location', 'Created_Corrected', 'Created_Corrected - Year'], value_vars=stages, var_name='stage', value_name='start')
    .dropna(subset=['start'])
    .assign(
        start=lambda d: pd.to_datetime(d['start']),
        stage=lambda d: d['stage'].str.removesuffix(' D&T')
    )
)

durations['order'] = durations['stage'].map(order)
durations = durations.sort_values(['ID', 'start', 'order'])
durations['end'] = durations.groupby('ID')['start'].shift(-1)
durations['days'] = (durations['end'] - durations['start']).dt.days
durations = durations[durations['stage'] != 'Complete']

status_output: pd.DataFrame = durations[['ID', 'Print Location', 'Created_Corrected', 'Created_Corrected - Year', 'stage', 'order', 'start', 'end', 'days']].reset_index(drop=True)

In [ ]:
SIZE_BINS = [0, 5, 10, 25, 50, 100, float('inf')]
SIZE_LABELS = ['< 5', '5-10', '10-25', '25-50', '50-100', '> 100']
DATE_COLS = ['Created', 'Created_Corrected', 'Created_Corrected - Year', 'Created_Corrected - Month']

def build_material_long(df):
    """One row per (job, machine, tier, material) with a quantity."""
    parts = []
    for col in df.columns:
        m = re.match(r'^(.+?) Qty T([123])$', col)
        if not m or col.startswith(('Total', 'Print Failure')):
            continue
        stem, tier = m[1], m[2]
        mat_col, color_col = f'{stem} T{tier}', f'{stem} Color T{tier}'
        role = 'Support' if 'Support' in stem else 'Model'
        machine = (re.split(r' (?:Model|Support)', stem)[0]
                    if re.search(r' (Model|Support)', stem) else stem.split(' ')[0])
        material = df[mat_col] if mat_col in df else pd.Series(stem, index=df.index)
        if mat_col in df and color_col in df:          # color typed into the material field
            material = material.where(material != df[color_col], df[f'{stem} T1'])
        parts.append(pd.DataFrame({
            'ID': df['ID'], 'Machine': machine, 'Print Location': df['Print Location'],
            'Material': material.fillna('Unspecified'), 'Role': role,
            'Qty': pd.to_numeric(df[col], errors='coerce')}))
    long = pd.concat(parts, ignore_index=True)
    long = long[long['Qty'].fillna(0) != 0].reset_index(drop=True)
    return long.merge(df[['ID'] + DATE_COLS], on='ID', how='left')

def build_job_size(df, long):
    """One row per (job, machine): total material + support qty, per-part average, size bin."""
    job = long.groupby(['ID', 'Machine'], as_index=False)['Qty'].sum().rename(columns={'Qty': 'Total Qty'})
    job = job.merge(df[['ID', 'Total Part Quantity'] + DATE_COLS], on='ID', how='left')
    parts = pd.to_numeric(job['Total Part Quantity'], errors='coerce')
    job['Avg Qty per Part'] = job['Total Qty'] / parts.where(parts > 0)
    job['Size Bin'] = pd.cut(job['Avg Qty per Part'], SIZE_BINS, labels=SIZE_LABELS, right=False)
    job['Size Bin Order'] = job['Size Bin'].cat.codes + 1
    return job


material_output = build_material_long(requests_df)
job_size_output = build_job_size(requests_df, material_output)

In [ ]:
machine_file = Path(output_folder / 'machines_data.csv')
machines_output.to_csv(machine_file, index=False)

status_file = Path(output_folder / 'status_data.csv')
status_output.to_csv(status_file, index=False)

material_file = Path(output_folder / 'material_qty_data.csv')
material_output.to_csv(material_file, index=False)

job_size_file = Path(output_folder / 'job_size_data.csv')
job_size_output.to_csv(job_size_file, index=False)